# Rare Cancer Drug Repurposing — Full Pipeline

**Drug Repurposing for Rare Cancers via GDSC-to-TCGA Transfer Learning**

Single self-contained notebook. Run every cell top to bottom.

**Part 1 — Data** (~20 min first run, cached after)
1. GDSC drug sensitivity (IC50) + expression (DepMap RNA-seq)
2. TCGA RNA-seq + clinical for 5 rare cancers
3. Gene harmonisation + quantile normalisation

**Part 2 — Models & Analysis** (~30 min)
4. Ridge regression training (5-fold CV)
5. Drug response imputation for TCGA patients
6. Mesothelioma survival analysis

In [ ]:
# ============================================================
# SETUP
# ============================================================
!pip install -q httpx openpyxl pyarrow lifelines

import csv as csv_mod
import gzip
import io
import json
import logging
import os
import tarfile
import warnings
import zipfile
from pathlib import Path

import httpx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# Persistent save directory (survives cell re-runs)
SAVE_DIR = "/content/rare_cancer_results"

for d in ["data/raw/gdsc", "data/raw/tcga", "data/processed", "data/results", "outputs/figures",
         f"{SAVE_DIR}/results", f"{SAVE_DIR}/figures"]:
    os.makedirs(d, exist_ok=True)

# Clear stale caches (keeps raw downloads)
for cache in ["data/processed/*", "data/raw/tcga/*.parquet"]:
    for f in Path(".").glob(cache):
        f.unlink()

print("Setup complete.")

In [ ]:
# Helper: save figure to both local and persistent dirs
def save_fig(name):
    plt.savefig(f'outputs/figures/{name}', dpi=150, bbox_inches='tight')
    plt.savefig(f'{SAVE_DIR}/figures/{name}', dpi=150, bbox_inches='tight')

def save_csv(df, name):
    df.to_csv(f'data/results/{name}', index=False)
    df.to_csv(f'{SAVE_DIR}/results/{name}', index=False)

def save_parquet(df, name):
    df.to_parquet(f'data/results/{name}')
    df.to_parquet(f'{SAVE_DIR}/results/{name}')

print('Save helpers ready.')

---
# Part 1 — Data Acquisition
## 1. GDSC Drug Sensitivity

In [ ]:
GDSC2_IC50_URL = (
    "https://cog.sanger.ac.uk/cancerrxgene/GDSC_release8.5/"
    "GDSC2_fitted_dose_response_27Oct23.xlsx"
)
ic50_path = Path("data/raw/gdsc/GDSC2_fitted_dose_response.xlsx")

if not ic50_path.exists():
    print("Downloading GDSC2 drug sensitivity data...")
    with httpx.stream("GET", GDSC2_IC50_URL, follow_redirects=True, timeout=300) as resp:
        resp.raise_for_status()
        with open(ic50_path, "wb") as f:
            for chunk in resp.iter_bytes(8192):
                f.write(chunk)
    print(f"Downloaded ({ic50_path.stat().st_size / 1e6:.1f} MB)")
else:
    print("Already downloaded.")

df = pd.read_excel(ic50_path)
col_map = {}
for col in df.columns:
    cu = col.upper().replace(" ", "_")
    if "COSMIC" in cu and "ID" in cu: col_map[col] = "COSMIC_ID"
    elif "DRUG" in cu and "NAME" in cu: col_map[col] = "DRUG_NAME"
    elif "LN_IC50" in cu: col_map[col] = "LN_IC50"
df = df.rename(columns=col_map).dropna(subset=["LN_IC50"])
df["COSMIC_ID"] = df["COSMIC_ID"].astype(int)

ic50 = df.pivot_table(index="COSMIC_ID", columns="DRUG_NAME", values="LN_IC50", aggfunc="median")
print(f"IC50 matrix: {ic50.shape[0]} cell lines x {ic50.shape[1]} drugs")
print(f"IC50 range: [{ic50.min().min():.2f}, {ic50.max().max():.2f}]")
ic50.head()

## 2. Cell Line Gene Expression (DepMap)

In [ ]:
DEPMAP_API_URL = "https://depmap.org/portal/api/download/files"
DEPMAP_RELEASE = "DepMap Public 24Q4"

def get_depmap_file_url(filename):
    resp = httpx.get(DEPMAP_API_URL, params={"release": DEPMAP_RELEASE}, timeout=60)
    resp.raise_for_status()
    for row in csv_mod.DictReader(io.StringIO(resp.text)):
        if row["filename"] == filename:
            return row["url"]
    raise RuntimeError(f"File '{filename}' not found")

# Download expression
expr_path = Path("data/raw/gdsc/depmap_expression.csv")
if not expr_path.exists():
    print("Downloading DepMap expression (~150 MB)...")
    url = get_depmap_file_url("OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv")
    with httpx.stream("GET", url, follow_redirects=True, timeout=600) as resp:
        resp.raise_for_status()
        with open(expr_path, "wb") as f:
            for chunk in resp.iter_bytes(8192): f.write(chunk)
    print(f"Downloaded ({expr_path.stat().st_size / 1e6:.1f} MB)")
else:
    print("Expression already downloaded.")

# Download sample info
si_path = Path("data/raw/gdsc/depmap_sample_info.csv")
if not si_path.exists():
    print("Downloading sample info...")
    url = get_depmap_file_url("Model.csv")
    with httpx.stream("GET", url, follow_redirects=True, timeout=120) as resp:
        resp.raise_for_status()
        with open(si_path, "wb") as f:
            for chunk in resp.iter_bytes(8192): f.write(chunk)
else:
    print("Sample info already downloaded.")

# Load and process
print("Loading expression matrix...")
raw = pd.read_csv(expr_path)
raw = raw[raw["IsDefaultEntryForModel"] == "Yes"]
print(f"  {len(raw)} cell lines after default-entry filter")

meta_names = ["SequencingID", "ModelConditionID", "ModelID", "IsDefaultEntryForMC", "IsDefaultEntryForModel"]
gene_cols = [c for c in raw.columns if c not in meta_names]
gdsc_expr = raw[gene_cols].copy()
gdsc_expr.index = raw["ModelID"].values
gdsc_expr.columns = [c.split(" (")[0] for c in gdsc_expr.columns]
gdsc_expr = gdsc_expr.loc[:, ~gdsc_expr.columns.duplicated(keep="first")]

# Map to COSMIC IDs
si = pd.read_csv(si_path)
id_map = si.dropna(subset=["COSMICID"]).set_index("ModelID")["COSMICID"].astype(int).to_dict()
gdsc_expr = gdsc_expr.loc[gdsc_expr.index.isin(id_map)]
gdsc_expr.index = gdsc_expr.index.map(id_map).astype(int)
gdsc_expr.index.name = "COSMIC_ID"

print(f"GDSC expression: {gdsc_expr.shape[0]} cell lines x {gdsc_expr.shape[1]} genes")
gdsc_expr.iloc[:3, :5]

## 3. TCGA Expression (5 Rare Cancers)

In [ ]:
RARE_CANCERS = ["TCGA-CHOL", "TCGA-MESO", "TCGA-UVM", "TCGA-ACC", "TCGA-THYM"]
GDC_FILES = "https://api.gdc.cancer.gov/files"
GDC_DATA = "https://api.gdc.cancer.gov/data"
tcga_cache = Path("data/raw/tcga/tcga_expression_combined.parquet")

if tcga_cache.exists():
    print("Loading cached TCGA expression...")
    tcga_expr = pd.read_parquet(tcga_cache)
else:
    all_samples, sample_projects = {}, {}
    for proj in RARE_CANCERS:
        filters = {"op":"and","content":[
            {"op":"in","content":{"field":"cases.project.project_id","value":[proj]}},
            {"op":"in","content":{"field":"data_category","value":["Transcriptome Profiling"]}},
            {"op":"in","content":{"field":"data_type","value":["Gene Expression Quantification"]}},
            {"op":"in","content":{"field":"analysis.workflow_type","value":["STAR - Counts"]}},
            {"op":"in","content":{"field":"access","value":["open"]}},
        ]}
        params = {"filters":json.dumps(filters),"fields":"file_id,file_name,cases.submitter_id,cases.samples.sample_type","format":"JSON","size":"500"}
        hits = httpx.get(GDC_FILES, params=params, timeout=60).json()["data"]["hits"]
        print(f"{proj}: {len(hits)} files")

        file_to_patient = {}
        for h in hits:
            cases = h.get("cases",[{}])
            pid = cases[0].get("submitter_id", h["file_id"]) if cases else h["file_id"]
            samples = cases[0].get("samples",[]) if cases else []
            if any("Primary" in s.get("sample_type","") or "Tumor" in s.get("sample_type","") for s in samples) or not samples:
                file_to_patient[h["file_id"]] = pid

        fids = list(file_to_patient.keys())
        for i in range(0, len(fids), 30):
            batch = fids[i:i+30]
            resp = httpx.post(GDC_DATA, json={"ids":batch}, timeout=600, headers={"Content-Type":"application/json"})
            resp.raise_for_status()
            try:
                with tarfile.open(fileobj=io.BytesIO(resp.content), mode="r:*") as tar:
                    for m in tar.getmembers():
                        if not (m.name.endswith(".tsv") or m.name.endswith(".tsv.gz")): continue
                        content = tar.extractfile(m).read()
                        if m.name.endswith(".gz"): content = gzip.decompress(content)
                        d = pd.read_csv(io.StringIO(content.decode()), sep="\t", comment="#")
                        fid = m.name.split("/")[0] if "/" in m.name else m.name
                        if fid not in file_to_patient: continue
                        cl = {c.lower():c for c in d.columns}
                        gc, fc = cl.get("gene_name"), cl.get("fpkm_unstranded", cl.get("tpm_unstranded", cl.get("unstranded")))
                        if gc and fc:
                            s = d.set_index(gc)[fc]
                            s = s[~s.index.duplicated(keep="first")]
                            all_samples[file_to_patient[fid]] = s
                            sample_projects[file_to_patient[fid]] = proj
            except tarfile.ReadError:
                content = gzip.decompress(resp.content) if resp.content[:2]==b"\x1f\x8b" else resp.content
                d = pd.read_csv(io.StringIO(content.decode()), sep="\t", comment="#")
                cl = {c.lower():c for c in d.columns}
                gc, fc = cl.get("gene_name"), cl.get("fpkm_unstranded", cl.get("tpm_unstranded"))
                if gc and fc and batch[0] in file_to_patient:
                    s = d.set_index(gc)[fc]; s = s[~s.index.duplicated(keep="first")]
                    all_samples[file_to_patient[batch[0]]] = s
                    sample_projects[file_to_patient[batch[0]]] = proj
        print(f"  Samples so far: {len(all_samples)}")

    tcga_expr = pd.DataFrame(all_samples).T
    tcga_expr = tcga_expr.apply(pd.to_numeric, errors="coerce")
    tcga_expr = np.log2(tcga_expr + 1)
    tcga_expr.insert(0, "project_id", pd.Series(sample_projects))
    tcga_expr.index.name = "patient_id"
    tcga_expr.to_parquet(tcga_cache)

print(f"TCGA: {tcga_expr.shape[0]} patients x {tcga_expr.shape[1]-1} genes")
print(tcga_expr["project_id"].value_counts())

## 4. TCGA Clinical Data

In [ ]:
GDC_CASES = "https://api.gdc.cancer.gov/cases"
clin_cache = Path("data/raw/tcga/tcga_clinical.parquet")

if clin_cache.exists():
    clinical = pd.read_parquet(clin_cache)
else:
    recs = []
    for proj in RARE_CANCERS:
        params = {
            "filters": json.dumps({"op":"in","content":{"field":"project.project_id","value":[proj]}}),
            "fields": "submitter_id,demographic.vital_status,demographic.days_to_death,demographic.gender,diagnoses.days_to_last_follow_up,diagnoses.tumor_stage,diagnoses.primary_diagnosis,diagnoses.age_at_diagnosis",
            "format": "JSON", "size": "500",
        }
        for c in httpx.get(GDC_CASES, params=params, timeout=60).json()["data"]["hits"]:
            demo = c.get("demographic",{})
            diag = (c.get("diagnoses") or [{}])[0]
            recs.append({"patient_id":c.get("submitter_id"),"project_id":proj,
                "vital_status":demo.get("vital_status"),"days_to_death":demo.get("days_to_death"),
                "days_to_last_follow_up":diag.get("days_to_last_follow_up"),
                "age_at_diagnosis":diag.get("age_at_diagnosis"),"gender":demo.get("gender")})
        print(f"{proj}: {len([r for r in recs if r['project_id']==proj])} patients")
    clinical = pd.DataFrame(recs).set_index("patient_id")
    clinical = clinical[~clinical.index.duplicated(keep="first")]
    clinical["os_days"] = clinical["days_to_death"].fillna(clinical["days_to_last_follow_up"])
    clinical["os_event"] = (clinical["vital_status"] == "Dead").astype(int)
    clinical.to_parquet(clin_cache)

print(f"Clinical: {len(clinical)} patients, {clinical['os_event'].sum()} deaths")
print(f"Median OS: {clinical['os_days'].median():.0f} days")

## 5. Gene Harmonisation

In [ ]:
# Standardise gene symbols
gdsc_expr.columns = gdsc_expr.columns.astype(str).str.strip().str.upper()
gdsc_expr = gdsc_expr.loc[:, ~gdsc_expr.columns.duplicated(keep="first")]
gdsc_expr = gdsc_expr.loc[:, gdsc_expr.columns != "NAN"]

tcga_meta = tcga_expr[["project_id"]].copy()
tcga_genes = tcga_expr.drop(columns=["project_id"])
tcga_genes = tcga_genes.loc[:, tcga_genes.columns.notna()]
tcga_genes = tcga_genes.loc[:, tcga_genes.columns.astype(str) != "nan"]
tcga_genes.columns = tcga_genes.columns.astype(str).str.strip().str.upper()
tcga_genes = tcga_genes.loc[:, ~tcga_genes.columns.duplicated(keep="first")]

# Overlap
common_genes = sorted(set(gdsc_expr.columns) & set(tcga_genes.columns))
exclude = ("LINC","LOC","MIR","SNORD","SNORA","SCARNA","RNU","RNA5","RNA18","RNA28")
common_genes = [g for g in common_genes if not g.startswith(exclude) and g not in ("NAN","NONE","")]

print(f"Common protein-coding genes: {len(common_genes)}")

# Verify
for g in ["TP53","EGFR","BRAF"]:
    print(f"  {g}: GDSC={'yes' if g in gdsc_expr.columns else 'NO'}, TCGA={'yes' if g in tcga_genes.columns else 'NO'}")

gdsc_harm = gdsc_expr[common_genes]
tcga_harm = tcga_genes[common_genes].copy()
tcga_harm.insert(0, "project_id", tcga_meta["project_id"])
print(f"GDSC harmonised: {gdsc_harm.shape}")
print(f"TCGA harmonised: {tcga_harm.shape}")

## 6. Quantile Normalisation

In [ ]:
# Low-expression filter
gdsc_means = gdsc_harm.mean(axis=0)
tcga_num = tcga_harm.drop(columns=["project_id"])
tcga_means = tcga_num.mean(axis=0)

keep_genes = [g for g in common_genes if gdsc_means.get(g,0)>=0.5 and tcga_means.get(g,0)>=0.5]
print(f"Genes after filter: {len(keep_genes)} (from {len(common_genes)})")

gdsc_filtered = gdsc_harm[keep_genes]
tcga_filtered = tcga_num[keep_genes]

# Quantile normalise TCGA -> GDSC reference
print("Quantile normalising...")
tcga_normalised = pd.DataFrame(index=tcga_filtered.index, columns=tcga_filtered.columns, dtype=float)
for j, gene in enumerate(keep_genes):
    tv = tcga_filtered[gene].values
    rv = np.sort(gdsc_filtered[gene].dropna().values)
    if len(rv) == 0:
        tcga_normalised[gene] = tv; continue
    ranks = stats.rankdata(tv, method="average")
    q = (ranks-1)/(len(ranks)-1) if len(ranks)>1 else ranks
    tcga_normalised[gene] = np.interp(q, np.linspace(0,1,len(rv)), rv)
    if (j+1)%3000==0: print(f"  {j+1}/{len(keep_genes)}...")

tcga_normalised.insert(0, "project_id", tcga_harm["project_id"])
print(f"Done. Normalised TCGA: {tcga_normalised.shape}")

In [ ]:
# Visualise normalisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sg = keep_genes[:500]
gf = gdsc_filtered[sg].values.flatten(); gf = gf[~np.isnan(gf)]
tn = tcga_normalised.drop(columns=["project_id"])[sg].values.flatten(); tn = tn[~np.isnan(tn)]
if len(gf)>0 and len(tn)>0:
    axes[0].hist(gf, bins=50, alpha=0.6, label="GDSC", density=True)
    axes[0].hist(tn, bins=50, alpha=0.6, label="TCGA (normalised)", density=True)
    axes[0].legend(); axes[0].set_title("After quantile normalisation")
    eg = sg[0]
    qg = np.quantile(gdsc_filtered[eg].dropna(), np.linspace(0,1,100))
    qt = np.quantile(tcga_normalised[eg].dropna(), np.linspace(0,1,100))
    axes[1].scatter(qg, qt, s=10, alpha=0.5)
    axes[1].plot([qg.min(),qg.max()],[qg.min(),qg.max()],"r--")
    axes[1].set_title(f"QQ plot ({eg})"); axes[1].set_xlabel("GDSC"); axes[1].set_ylabel("TCGA")
plt.tight_layout()
plt.savefig("outputs/figures/normalisation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Save processed data
gdsc_filtered.to_parquet("data/processed/gdsc_expression.parquet")
ic50.to_parquet("data/processed/gdsc_ic50.parquet")
tcga_normalised.to_parquet("data/processed/tcga_expression_normalised.parquet")
clinical.to_parquet("data/processed/tcga_clinical.parquet")

print("=" * 50)
print("PART 1 COMPLETE")
print(f"GDSC: {gdsc_filtered.shape[0]} cell lines, {ic50.shape[1]} drugs, {gdsc_filtered.shape[1]} genes")
print(f"TCGA: {tcga_normalised.shape[0]} patients")
for ct in RARE_CANCERS:
    n = (tcga_normalised["project_id"]==ct).sum()
    d = clinical.loc[clinical["project_id"]==ct, "os_event"].sum()
    print(f"  {ct}: {n} samples, {d} deaths")

# Also save to persistent directory
import shutil
for f in Path('data/processed').glob('*'):
    shutil.copy2(f, f'{SAVE_DIR}/results/{f.name}')
print(f'Backed up to {SAVE_DIR}/results/')

---
# Part 2 — Model Training & Analysis
## 8. Ridge Regression Training

In [ ]:
# Variance filter: keep top 5000 most variable genes (reduces runtime ~6x)
from sklearn.linear_model import Ridge

variances = gdsc_filtered.var(axis=0).sort_values(ascending=False)
top_genes = variances.head(5000).index.tolist()
gdsc_var = gdsc_filtered[top_genes]
print(f"Variance filter: {gdsc_filtered.shape[1]} -> {len(top_genes)} genes")

common_cells = sorted(set(gdsc_var.index.astype(int)) & set(ic50.index.astype(int)))
print(f"Cell lines with both expression and IC50: {len(common_cells)}")

X_all = gdsc_var.loc[common_cells].values.astype(np.float32)
gene_names = list(gdsc_var.columns)
alphas = np.logspace(-1, 5, 20)
MIN_SAMPLES, R2_THRESHOLD = 30, 0.1

models, scalers, cv_results = {}, {}, []
drugs = ic50.columns.tolist()
print(f"Training ridge for {len(drugs)} drugs (~10-15 min)...")

import time
t0 = time.time()
for i, drug in enumerate(drugs):
    y = ic50.loc[common_cells, drug]
    valid = y.notna()
    if valid.sum() < MIN_SAMPLES: continue
    X = X_all[valid.values]
    y_vals = y[valid].values.astype(np.float64)
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X)

    # Fit RidgeCV to select best alpha
    model = RidgeCV(alphas=alphas, cv=5, scoring="r2")
    model.fit(X_sc, y_vals)

    # Get CV R² at the chosen alpha using a lightweight Ridge (no alpha search)
    r2_scores = cross_val_score(Ridge(alpha=model.alpha_), X_sc, y_vals, cv=5, scoring="r2")
    r2 = float(r2_scores.mean())

    cv_results.append({"drug":drug, "n_samples":int(valid.sum()),
        "cv_r2_mean":r2, "cv_r2_std":float(r2_scores.std()),
        "best_alpha":model.alpha_, "predictable":r2>=R2_THRESHOLD})
    if r2 >= R2_THRESHOLD:
        models[drug] = model; scalers[drug] = scaler
    if (i+1)%50==0:
        elapsed = time.time() - t0
        print(f"  {i+1}/{len(drugs)} ({elapsed:.0f}s elapsed)...")

cv_df = pd.DataFrame(cv_results).sort_values("cv_r2_mean", ascending=False)
n_pred = cv_df["predictable"].sum()
elapsed = time.time() - t0
print(f"\nDone in {elapsed:.0f}s! Predictable drugs (R2>{R2_THRESHOLD}): {n_pred}/{len(cv_df)}")
cv_df[cv_df["predictable"]].head(10)[["drug","cv_r2_mean","cv_r2_std","n_samples"]]

In [ ]:
# CV performance plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(cv_df["cv_r2_mean"], bins=30, edgecolor="black", alpha=0.7)
axes[0].axvline(R2_THRESHOLD, color="red", linestyle="--", label=f"threshold ({R2_THRESHOLD})")
axes[0].set_xlabel("CV R\u00b2"); axes[0].set_ylabel("Drugs"); axes[0].legend()
p = cv_df[cv_df["predictable"]]; np_ = cv_df[~cv_df["predictable"]]
axes[1].scatter(np_["n_samples"], np_["cv_r2_mean"], alpha=0.4, s=20, color="gray", label="Not predictable")
axes[1].scatter(p["n_samples"], p["cv_r2_mean"], alpha=0.6, s=20, color="steelblue", label="Predictable")
axes[1].axhline(R2_THRESHOLD, color="red", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Cell lines"); axes[1].set_ylabel("CV R\u00b2"); axes[1].legend()
plt.tight_layout()
plt.savefig("outputs/figures/ridge_cv_performance.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Impute Drug Responses

In [ ]:
tcga_norm_num = tcga_normalised.drop(columns=["project_id"])
X_tcga = tcga_norm_num.reindex(columns=gene_names, fill_value=0).values.astype(np.float32)

imputed = {}
for drug, model in models.items():
    imputed[drug] = model.predict(scalers[drug].transform(X_tcga))

imputed = pd.DataFrame(imputed, index=tcga_normalised.index)
imputed.insert(0, "project_id", tcga_normalised["project_id"])
imputed.index.name = "patient_id"

print(f"Imputed: {imputed.shape[0]} patients x {len(models)} drugs")
print(imputed["project_id"].value_counts())

In [ ]:
# Heatmap (Figure 1 candidate)
drug_cols = [c for c in imputed.columns if c != "project_id"]
mean_by_cancer = imputed.groupby("project_id")[drug_cols].mean()
top30 = mean_by_cancer.var(axis=0).nlargest(30).index.tolist()

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(mean_by_cancer[top30].T, cmap="RdBu_r", center=0,
            xticklabels=[c.replace("TCGA-","") for c in mean_by_cancer.index], ax=ax)
ax.set_title("Mean imputed drug sensitivity across rare cancer types\n(lower = more sensitive)")
plt.tight_layout()
plt.savefig("outputs/figures/imputed_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Survival Analysis (Mesothelioma)

In [ ]:
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from statsmodels.stats.multitest import multipletests

meso_imp = imputed[imputed["project_id"]=="TCGA-MESO"].drop(columns=["project_id"])
meso_clin = clinical.loc[clinical.index.isin(meso_imp.index)]
pts = sorted(set(meso_imp.index) & set(meso_clin.index))
meso_imp, meso_clin = meso_imp.loc[pts], meso_clin.loc[pts]
valid = meso_clin["os_days"].notna() & (meso_clin["os_days"]>0)
meso_imp, meso_clin = meso_imp.loc[valid], meso_clin.loc[valid]
print(f"MESO patients with survival: {len(meso_clin)}")

surv = []
for drug in meso_imp.columns:
    y = meso_imp[drug]
    if y.notna().sum()<20: continue
    sens = y <= y.median()
    if sens.sum()<10 or (~sens).sum()<10: continue
    lr = logrank_test(meso_clin.loc[sens,"os_days"],meso_clin.loc[~sens,"os_days"],meso_clin.loc[sens,"os_event"],meso_clin.loc[~sens,"os_event"])
    try:
        cph = CoxPHFitter()
        cph.fit(pd.DataFrame({"os_days":meso_clin["os_days"],"os_event":meso_clin["os_event"],"drug":y}).dropna(), "os_days","os_event")
        hr, cp = np.exp(cph.params_["drug"]), cph.summary.loc["drug","p"]
    except: hr, cp = np.nan, np.nan
    surv.append({"drug":drug,"logrank_p":lr.p_value,"cox_hr":hr,"cox_p":cp,"n_sens":int(sens.sum()),"n_res":int((~sens).sum())})

meso_surv = pd.DataFrame(surv).sort_values("logrank_p")
if meso_surv.empty:
    print("No drugs met survival analysis criteria.")
    meso_surv["fdr_q"] = []
else:
    _, fdr, _, _ = multipletests(meso_surv["logrank_p"], method="fdr_bh")
    meso_surv["fdr_q"] = fdr
    print(f"Drugs tested: {len(meso_surv)}")
    print(f"Significant (FDR<0.05): {(meso_surv['fdr_q']<0.05).sum()}")
    print(f"Nominal (p<0.05): {(meso_surv['logrank_p']<0.05).sum()}")
    meso_surv.head(10)

In [ ]:
# KM curves for top 4 drugs
if not meso_surv.empty and len(meso_surv) >= 4:
    top4 = meso_surv.head(4)["drug"].tolist()
elif not meso_surv.empty:
    top4 = meso_surv["drug"].tolist()
else:
    top4 = []

if top4:
    n_plots = len(top4)
    ncols = min(n_plots, 2)
    nrows = (n_plots + 1) // 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 5*nrows), squeeze=False)
    for idx, drug in enumerate(top4):
        ax = axes[idx // ncols, idx % ncols]
        y = meso_imp[drug]; sens = y <= y.median()
        kmf = KaplanMeierFitter()
        kmf.fit(meso_clin.loc[sens,"os_days"],meso_clin.loc[sens,"os_event"],label=f"Sensitive (n={sens.sum()})")
        kmf.plot_survival_function(ax=ax, ci_show=True)
        kmf.fit(meso_clin.loc[~sens,"os_days"],meso_clin.loc[~sens,"os_event"],label=f"Resistant (n={(~sens).sum()})")
        kmf.plot_survival_function(ax=ax, ci_show=True)
        lr = logrank_test(meso_clin.loc[sens,"os_days"],meso_clin.loc[~sens,"os_days"],meso_clin.loc[sens,"os_event"],meso_clin.loc[~sens,"os_event"])
        ax.set_title(drug); ax.set_xlabel("Days"); ax.set_ylabel("Survival")
        ax.text(0.95,0.95,f"p={lr.p_value:.4f}",transform=ax.transAxes,ha="right",va="top",fontsize=10,bbox=dict(boxstyle="round",facecolor="white",alpha=0.8))
    plt.suptitle("Mesothelioma: Top drug-survival associations", fontsize=14)
    plt.tight_layout()
    plt.savefig("outputs/figures/meso_km_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No drugs to plot.")

## 11. Save Results

In [ ]:
imputed.to_parquet("data/results/imputed_drug_responses.parquet")
imputed.to_parquet(f"{SAVE_DIR}/results/imputed_drug_responses.parquet")
cv_df.to_csv("data/results/ridge_cv_scores.csv", index=False)
cv_df.to_csv(f"{SAVE_DIR}/results/ridge_cv_scores.csv", index=False)
meso_surv.to_csv("data/results/meso_survival_results.csv", index=False)
meso_surv.to_csv(f"{SAVE_DIR}/results/meso_survival_results.csv", index=False)

print("=" * 50)
print("PIPELINE COMPLETE")
print("=" * 50)
print(f"Data: {len(common_cells)} cell lines, {ic50.shape[1]} drugs, {len(gene_names)} genes (variance-filtered)")
print(f"TCGA: {tcga_normalised.shape[0]} patients across {len(RARE_CANCERS)} cancers")
print(f"Models: {n_pred} predictable drugs (CV R2>{R2_THRESHOLD})")
print(f"Imputed: {imputed.shape[0]} patients x {len(models)} drugs")
if not meso_surv.empty:
    print(f"MESO: {(meso_surv['logrank_p']<0.05).sum()} drugs nominally associated with survival")
print(f"\nSaved:")
for d in ["data/processed","data/results"]:
    for f in sorted(Path(d).glob("*")):
        print(f"  {f}: {f.stat().st_size/1e6:.1f} MB")

---
# Part 3 — Extended Analysis
## 12. Survival Analysis (All 5 Cancer Types)

In [ ]:
# Run survival analysis for all 5 rare cancer types
all_surv = []
for ct in RARE_CANCERS:
    ct_imp = imputed[imputed['project_id']==ct].drop(columns=['project_id'])
    ct_clin = clinical.loc[clinical.index.isin(ct_imp.index)]
    pts = sorted(set(ct_imp.index) & set(ct_clin.index))
    if len(pts) < 20: print(f'{ct}: too few patients ({len(pts)})'); continue
    ct_imp, ct_clin = ct_imp.loc[pts], ct_clin.loc[pts]
    valid = ct_clin['os_days'].notna() & (ct_clin['os_days']>0)
    ct_imp, ct_clin = ct_imp.loc[valid], ct_clin.loc[valid]
    print(f'{ct}: {len(ct_clin)} patients with survival data')

    for drug in ct_imp.columns:
        y = ct_imp[drug]
        if y.notna().sum()<15: continue
        sens = y <= y.median()
        if sens.sum()<7 or (~sens).sum()<7: continue
        lr = logrank_test(ct_clin.loc[sens,'os_days'],ct_clin.loc[~sens,'os_days'],
                         ct_clin.loc[sens,'os_event'],ct_clin.loc[~sens,'os_event'])
        try:
            cph = CoxPHFitter()
            cph.fit(pd.DataFrame({'os_days':ct_clin['os_days'],'os_event':ct_clin['os_event'],'drug':y}).dropna(),'os_days','os_event')
            hr = np.exp(cph.params_['drug'])
        except: hr = np.nan
        all_surv.append({'cancer':ct,'drug':drug,'logrank_p':lr.p_value,'cox_hr':hr,
                         'n_sens':int(sens.sum()),'n_res':int((~sens).sum())})

all_surv_df = pd.DataFrame(all_surv)

# FDR correction within each cancer type
all_surv_df['fdr_q'] = np.nan
for ct in all_surv_df['cancer'].unique():
    mask = all_surv_df['cancer']==ct
    _, fdr, _, _ = multipletests(all_surv_df.loc[mask,'logrank_p'], method='fdr_bh')
    all_surv_df.loc[mask,'fdr_q'] = fdr

# Summary
print('\n' + '='*50)
print('SURVIVAL ANALYSIS SUMMARY')
print('='*50)
for ct in RARE_CANCERS:
    ct_data = all_surv_df[all_surv_df['cancer']==ct]
    if ct_data.empty: continue
    n_fdr = (ct_data['fdr_q']<0.05).sum()
    n_nom = (ct_data['logrank_p']<0.05).sum()
    print(f'{ct}: {len(ct_data)} drugs tested, {n_fdr} FDR-sig, {n_nom} nominal (p<0.05)')

all_surv_df.to_csv('data/results/all_cancer_survival.csv', index=False)
all_surv_df.to_csv(f'{SAVE_DIR}/results/all_cancer_survival.csv', index=False)
print('\nTop 5 per cancer type:')
for ct in RARE_CANCERS:
    ct_data = all_surv_df[all_surv_df['cancer']==ct].sort_values('logrank_p').head(5)
    if ct_data.empty: continue
    print(f'\n{ct}:')
    print(ct_data[['drug','logrank_p','fdr_q','cox_hr']].to_string(index=False))

## 13. IDWAS — Pharmacogenomic Associations

In [ ]:
# IDWAS: correlate gene expression with imputed drug sensitivity per cancer type
# Focus on MESO first (strongest results), then all cancers
tcga_norm_num = tcga_normalised.drop(columns=['project_id'])

def run_idwas(imp_df, expr_df, cancer_type, top_n_drugs=20):
    """Correlate each gene's expression with each drug's imputed IC50."""
    mask = imp_df['project_id'] == cancer_type
    imp = imp_df.loc[mask].drop(columns=['project_id'])
    expr = expr_df.loc[mask] if 'project_id' not in expr_df.columns else expr_df.loc[mask]
    common = sorted(set(imp.index) & set(expr.index))
    if len(common) < 15: return pd.DataFrame()
    imp, expr = imp.loc[common], expr.loc[common]

    # Use top drugs by survival significance for this cancer
    ct_surv = all_surv_df[all_surv_df['cancer']==cancer_type].sort_values('logrank_p')
    top_drugs = ct_surv.head(top_n_drugs)['drug'].tolist()
    if not top_drugs: top_drugs = imp.columns[:top_n_drugs].tolist()

    results = []
    for drug in top_drugs:
        if drug not in imp.columns: continue
        y = imp[drug].values
        for gene in expr.columns[:5000]:  # limit genes for speed
            x = expr[gene].values
            valid = ~(np.isnan(x) | np.isnan(y))
            if valid.sum() < 15: continue
            r, p = stats.pearsonr(x[valid], y[valid])
            results.append({'drug':drug,'gene':gene,'pearson_r':r,'p_value':p})

    if not results: return pd.DataFrame()
    df = pd.DataFrame(results)
    _, fdr, _, _ = multipletests(df['p_value'], method='fdr_bh')
    df['fdr_q'] = fdr
    return df.sort_values('p_value')

print('Running IDWAS for MESO (top 20 survival-associated drugs x 5000 genes)...')
meso_idwas = run_idwas(imputed, tcga_norm_num, 'TCGA-MESO', top_n_drugs=20)
n_sig = (meso_idwas['fdr_q']<0.05).sum() if not meso_idwas.empty else 0
print(f'MESO IDWAS: {len(meso_idwas)} tests, {n_sig} FDR-significant associations')
if not meso_idwas.empty:
    print('\nTop 20 gene-drug associations:')
    print(meso_idwas.head(20)[['drug','gene','pearson_r','p_value','fdr_q']].to_string(index=False))

In [ ]:
# Run IDWAS for all cancer types
all_idwas = {}
for ct in RARE_CANCERS:
    print(f'Running IDWAS for {ct}...')
    result = run_idwas(imputed, tcga_norm_num, ct, top_n_drugs=10)
    if not result.empty:
        all_idwas[ct] = result
        n_sig = (result['fdr_q']<0.05).sum()
        print(f'  {len(result)} tests, {n_sig} FDR-significant')
    else:
        print(f'  No results (too few patients?)')

# Combine and save
if all_idwas:
    combined_idwas = pd.concat([df.assign(cancer=ct) for ct, df in all_idwas.items()])
    combined_idwas.to_csv('data/results/idwas_all_cancers.csv', index=False)
    combined_idwas.to_csv(f'{SAVE_DIR}/results/idwas_all_cancers.csv', index=False)
    print(f'\nSaved {len(combined_idwas)} total IDWAS results')

## 14. Volcano Plot (MESO IDWAS)

In [ ]:
# Volcano plot for MESO IDWAS — top drug
if not meso_idwas.empty:
    top_drug = meso_idwas['drug'].value_counts().index[0]
    drug_data = meso_idwas[meso_idwas['drug']==top_drug].copy()
    drug_data['-log10p'] = -np.log10(drug_data['p_value'].clip(lower=1e-50))

    fig, ax = plt.subplots(figsize=(10, 7))
    sig = drug_data['fdr_q'] < 0.05
    ax.scatter(drug_data.loc[~sig,'pearson_r'], drug_data.loc[~sig,'-log10p'],
               s=8, alpha=0.3, color='gray', label='Not significant')
    ax.scatter(drug_data.loc[sig,'pearson_r'], drug_data.loc[sig,'-log10p'],
               s=15, alpha=0.7, color='crimson', label=f'FDR < 0.05 (n={sig.sum()})')

    # Label top genes
    for _, row in drug_data.head(10).iterrows():
        ax.annotate(row['gene'], (row['pearson_r'], row['-log10p']),
                    fontsize=7, alpha=0.8, ha='center', va='bottom')

    ax.axhline(-np.log10(0.05), color='blue', linestyle='--', alpha=0.3, label='p=0.05')
    ax.set_xlabel('Pearson r (gene expression vs imputed IC50)')
    ax.set_ylabel('-log10(p-value)')
    ax.set_title(f'MESO IDWAS: {top_drug}')
    ax.legend()
    plt.tight_layout()
    plt.savefig('outputs/figures/meso_volcano.png', dpi=150, bbox_inches='tight')
    plt.savefig(f'{SAVE_DIR}/figures/meso_volcano.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No IDWAS results to plot.')

## 15. KM Curves — Best Drug per Cancer Type

In [ ]:
# KM curve for best drug in each cancer type
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
for ax, ct in zip(axes, RARE_CANCERS):
    ct_data = all_surv_df[all_surv_df['cancer']==ct].sort_values('logrank_p')
    if ct_data.empty:
        ax.set_title(f'{ct.replace("TCGA-","")}: no data'); continue
    best_drug = ct_data.iloc[0]['drug']
    best_p = ct_data.iloc[0]['logrank_p']

    ct_imp = imputed[imputed['project_id']==ct].drop(columns=['project_id'])
    ct_clin = clinical.loc[clinical.index.isin(ct_imp.index)]
    pts = sorted(set(ct_imp.index) & set(ct_clin.index))
    ct_imp, ct_clin = ct_imp.loc[pts], ct_clin.loc[pts]
    valid = ct_clin['os_days'].notna() & (ct_clin['os_days']>0)
    ct_imp, ct_clin = ct_imp.loc[valid], ct_clin.loc[valid]

    y = ct_imp[best_drug]; sens = y <= y.median()
    kmf = KaplanMeierFitter()
    kmf.fit(ct_clin.loc[sens,'os_days'],ct_clin.loc[sens,'os_event'],label=f'Sens (n={sens.sum()})')
    kmf.plot_survival_function(ax=ax, ci_show=True)
    kmf.fit(ct_clin.loc[~sens,'os_days'],ct_clin.loc[~sens,'os_event'],label=f'Res (n={(~sens).sum()})')
    kmf.plot_survival_function(ax=ax, ci_show=True)
    ax.set_title(f'{ct.replace("TCGA-","")}\n{best_drug} (p={best_p:.4f})', fontsize=10)
    ax.set_xlabel('Days'); ax.set_ylabel('Survival')
    ax.legend(fontsize=7)

plt.suptitle('Best survival-associated drug per rare cancer type', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/figures/all_cancer_km.png', dpi=150, bbox_inches='tight')
plt.savefig(f'{SAVE_DIR}/figures/all_cancer_km.png', dpi=150, bbox_inches='tight')
plt.show()

## 17. GSEA Pathway Enrichment

In [ ]:
!pip install -q gseapy shap
import gseapy as gp

gsea_results = {}
for ct in RARE_CANCERS:
    ct_surv = all_surv_df[all_surv_df['cancer']==ct].sort_values('logrank_p')
    if ct_surv.empty: continue
    top_drug = ct_surv.iloc[0]['drug']
    if ct not in all_idwas or all_idwas[ct].empty: continue
    drug_idwas = all_idwas[ct][all_idwas[ct]['drug']==top_drug]
    if len(drug_idwas) < 50:
        drug_idwas = all_idwas[ct][all_idwas[ct]['drug']==all_idwas[ct]['drug'].value_counts().index[0]]
        top_drug = drug_idwas['drug'].iloc[0]
    if len(drug_idwas) < 50: continue
    gene_ranking = drug_idwas.set_index('gene')['pearson_r'].sort_values(ascending=False)
    gene_ranking = gene_ranking[~gene_ranking.index.duplicated(keep='first')]
    print(f'{ct}: GSEA for {top_drug} ({len(gene_ranking)} genes)...')
    try:
        pre_res = gp.prerank(rnk=gene_ranking, gene_sets='MSigDB_Hallmark_2020',
                             organism='human', permutation_num=500, outdir=None, seed=42, verbose=False)
        res_df = pre_res.res2d.copy()
        if not res_df.empty:
            res_df['cancer'], res_df['drug'] = ct, top_drug
            gsea_results[ct] = res_df
            n_sig = (res_df['FDR q-val'].astype(float) < 0.25).sum()
            print(f'  {n_sig} enriched pathways (FDR < 0.25)')
    except Exception as e:
        print(f'  Failed: {e}')

all_gsea = pd.concat(gsea_results.values(), ignore_index=True) if gsea_results else pd.DataFrame()
if not all_gsea.empty:
    all_gsea.to_csv(f'{SAVE_DIR}/results/gsea_all_cancers.csv', index=False)
    print(f'Saved {len(all_gsea)} GSEA results')


## 18. SHAP Biomarkers

In [ ]:
import shap

top5_drugs = cv_df[cv_df['predictable']].head(5)['drug'].tolist()
meso_expr = tcga_normalised[tcga_normalised['project_id']=='TCGA-MESO'].drop(columns=['project_id'])
X_meso = meso_expr.reindex(columns=gene_names, fill_value=0).values.astype(np.float32)

shap_results = {}
for drug in top5_drugs:
    if drug not in models: continue
    X_sc = scalers[drug].transform(X_meso)
    explainer = shap.LinearExplainer(models[drug], X_sc)
    sv = explainer.shap_values(X_sc)
    importance = pd.DataFrame({'gene':gene_names, 'mean_abs_shap':np.abs(sv).mean(axis=0),
        'mean_shap':sv.mean(axis=0),
        'direction':np.where(sv.mean(axis=0)>0,'resistance','sensitivity')
    }).sort_values('mean_abs_shap', ascending=False)
    importance.insert(0, 'drug', drug)
    shap_results[drug] = importance
    print(f'{drug}: top gene = {importance.iloc[0]["gene"]} ({importance.iloc[0]["direction"]})')

all_shap = pd.concat([df.head(20) for df in shap_results.values()], ignore_index=True)
all_shap.to_csv(f'{SAVE_DIR}/results/shap_biomarkers.csv', index=False)
print(f'Saved SHAP for {len(shap_results)} drugs')

# Bar plot for top 2
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, drug in zip(axes, list(shap_results.keys())[:2]):
    imp = shap_results[drug].head(15)
    colors = ['crimson' if d=='resistance' else 'steelblue' for d in imp['direction']]
    ax.barh(range(len(imp)), imp['mean_abs_shap'].values, color=colors, alpha=0.8)
    ax.set_yticks(range(len(imp))); ax.set_yticklabels(imp['gene'].values, fontsize=9)
    ax.set_xlabel('Mean |SHAP|'); ax.set_title(f'{drug}'); ax.invert_yaxis()
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color='crimson',label='resistance'),Patch(color='steelblue',label='sensitivity')],fontsize=8)
plt.tight_layout()
plt.savefig('outputs/figures/shap_biomarkers.png', dpi=150, bbox_inches='tight')
plt.savefig(f'{SAVE_DIR}/figures/shap_biomarkers.png', dpi=150, bbox_inches='tight')
plt.show()


## 19. Validation

In [ ]:
# Permutation test (100 perms)
import time
print('Permutation test (100 perms) for MESO...')
t0 = time.time()
n_perms = 100
perm_fdr_counts = []
meso_mask_p = imputed['project_id']=='TCGA-MESO'
meso_imp_p = imputed.loc[meso_mask_p].drop(columns=['project_id'])
meso_clin_p = clinical.loc[clinical.index.isin(meso_imp_p.index)]
pts_p = sorted(set(meso_imp_p.index) & set(meso_clin_p.index))
meso_imp_p, meso_clin_p = meso_imp_p.loc[pts_p], meso_clin_p.loc[pts_p]
valid_p = meso_clin_p['os_days'].notna() & (meso_clin_p['os_days']>0)
meso_imp_p, meso_clin_p = meso_imp_p.loc[valid_p], meso_clin_p.loc[valid_p]
rng = np.random.RandomState(42)
for perm in range(n_perms):
    perm_surv = []
    sc = meso_clin_p.copy()
    si = rng.permutation(len(sc))
    sc['os_days'], sc['os_event'] = sc['os_days'].values[si], sc['os_event'].values[si]
    for drug in meso_imp_p.columns:
        y = meso_imp_p[drug]
        if y.notna().sum()<20: continue
        sens = y<=y.median()
        if sens.sum()<10 or (~sens).sum()<10: continue
        lr = logrank_test(sc.loc[sens,'os_days'],sc.loc[~sens,'os_days'],sc.loc[sens,'os_event'],sc.loc[~sens,'os_event'])
        perm_surv.append(lr.p_value)
    if perm_surv:
        _, pf, _, _ = multipletests(perm_surv, method='fdr_bh')
        perm_fdr_counts.append((pf<0.05).sum())
    else: perm_fdr_counts.append(0)
    if (perm+1)%25==0: print(f'  {perm+1}/{n_perms} ({time.time()-t0:.0f}s)')

real_count = (meso_surv['fdr_q']<0.05).sum()
perm_p = np.mean([c>=real_count for c in perm_fdr_counts])
print(f'\nReal: {real_count} FDR-sig, Permuted mean: {np.mean(perm_fdr_counts):.1f}, p={perm_p:.4f}')

fig, ax = plt.subplots(figsize=(8,5))
ax.hist(perm_fdr_counts, bins=20, edgecolor='black', alpha=0.7)
ax.axvline(real_count, color='red', linewidth=2, linestyle='--', label=f'Real ({real_count})')
ax.set_xlabel('FDR-sig drugs'); ax.set_ylabel('Frequency'); ax.legend()
ax.set_title(f'Permutation test (p={perm_p:.4f})')
plt.tight_layout()
plt.savefig('outputs/figures/permutation_test.png', dpi=150, bbox_inches='tight')
plt.savefig(f'{SAVE_DIR}/figures/permutation_test.png', dpi=150, bbox_inches='tight')
plt.show()

# Split-half validation
print('\nSplit-half validation...')
rng2 = np.random.RandomState(123)
idx = rng2.permutation(len(meso_imp_p))
h = len(idx)//2
s1i, s2i = meso_imp_p.index[idx[:h]], meso_imp_p.index[idx[h:]]
def qsurv(imp, clin):
    r = {}
    for d in imp.columns:
        y=imp[d]; s=y<=y.median()
        if s.sum()<5 or (~s).sum()<5: continue
        lr=logrank_test(clin.loc[s,'os_days'],clin.loc[~s,'os_days'],clin.loc[s,'os_event'],clin.loc[~s,'os_event'])
        r[d] = -np.log10(max(lr.p_value,1e-50))
    return pd.Series(r)
r1, r2 = qsurv(meso_imp_p.loc[s1i],meso_clin_p.loc[s1i]), qsurv(meso_imp_p.loc[s2i],meso_clin_p.loc[s2i])
cd = sorted(set(r1.index)&set(r2.index))
if len(cd)>=20:
    r, p = stats.pearsonr(r1[cd], r2[cd])
    print(f'Split-half: r={r:.3f}, p={p:.2e} ({len(cd)} drugs)')
    fig, ax = plt.subplots(figsize=(7,7))
    ax.scatter(r1[cd], r2[cd], s=20, alpha=0.5)
    ax.set_xlabel('Split 1: -log10(p)'); ax.set_ylabel('Split 2: -log10(p)')
    ax.set_title(f'Split-half (r={r:.3f}, p={p:.2e})')
    lims=[0,max(r1[cd].max(),r2[cd].max())]; ax.plot(lims,lims,'r--',alpha=0.3)
    plt.tight_layout()
    plt.savefig('outputs/figures/split_half.png', dpi=150, bbox_inches='tight')
    plt.savefig(f'{SAVE_DIR}/figures/split_half.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# ============================================================
# SAVE ALL RESULTS AND DOWNLOAD
# ============================================================
import shutil

print('Saving all results...')

# Save everything that exists
saves = [
    ('imputed', 'imputed_drug_responses.parquet', 'parquet'),
    ('cv_df', 'ridge_cv_scores.csv', 'csv'),
    ('meso_surv', 'meso_survival_results.csv', 'csv'),
    ('all_surv_df', 'all_cancer_survival.csv', 'csv'),
]
# Optional saves (from GSEA/SHAP if they were run)
optional = [
    ('all_shap', 'shap_biomarkers.csv', 'csv'),
    ('all_gsea', 'gsea_all_cancers.csv', 'csv'),
    ('combined_idwas', 'idwas_all_cancers.csv', 'csv'),
]

for var_name, fname, fmt in saves + optional:
    if var_name not in dir() and var_name not in globals():
        continue
    obj = globals().get(var_name)
    if obj is None:
        continue
    try:
        if fmt == 'parquet':
            obj.to_parquet(f'{SAVE_DIR}/results/{fname}')
        else:
            obj.to_csv(f'{SAVE_DIR}/results/{fname}', index=False)
        print(f'  Saved {fname}')
    except Exception as e:
        print(f'  Skipped {fname}: {e}')

# Copy any figures
for src_dir in ['outputs/figures']:
    if os.path.exists(src_dir):
        for f in Path(src_dir).glob('*.png'):
            shutil.copy2(f, f'{SAVE_DIR}/figures/{f.name}')
            print(f'  Saved {f.name}')

# List everything
print(f'\nAll files in {SAVE_DIR}:')
for root, dirs, fnames in os.walk(SAVE_DIR):
    for f in sorted(fnames):
        fp = os.path.join(root, f)
        print(f'  {os.path.relpath(fp, SAVE_DIR)}: {os.path.getsize(fp)/1e6:.1f} MB')

# Zip and download
print('\nZipping...')
shutil.make_archive('/content/rare_cancer_results', 'zip', SAVE_DIR)
print('Downloading...')
from google.colab import files
files.download('/content/rare_cancer_results.zip')
